# Notebook 2: Exploratory Data Analysis
**DS340W Capstone — Group 60**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
base_path = '/content/drive/MyDrive/capstone_project'
merged = pd.read_csv(f'{base_path}/processed_data/final_dataset.csv')
merged['date'] = pd.to_datetime(merged['date'])
colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']
print(f"Loaded {len(merged)} rows")

In [ ]:
# Crime over time
daily_total = merged.groupby('date')[['assault','criminal_damage','robbery','theft']].sum()
fig, ax = plt.subplots(figsize=(16,6))
daily_total.plot(ax=ax, alpha=0.7)
ax.set_title('Daily Crime Counts in NYC (2015-2020)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Number of Crimes')
plt.tight_layout()
plt.savefig(f'{base_path}/outputs/crime_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Crime by type
totals = merged[['assault','criminal_damage','robbery','theft']].sum()
fig, ax = plt.subplots(figsize=(10,6))
totals.plot(kind='bar', ax=ax, color=colors, edgecolor='black')
ax.set_title('Total Crime Counts by Type (2015-2020)', fontsize=16, fontweight='bold')
ax.set_xticklabels(['Assault','Criminal Damage','Robbery','Theft'], rotation=0)
for i, v in enumerate(totals): ax.text(i, v+v*0.01, f'{int(v):,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{base_path}/outputs/crime_by_type.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Monthly seasonality
merged['month_num'] = merged['date'].dt.month
monthly = merged.groupby('month_num')[['assault','criminal_damage','robbery','theft']].mean()
fig, axes = plt.subplots(2,2, figsize=(14,10))
for idx, (crime, title) in enumerate(zip(['assault','criminal_damage','robbery','theft'], ['Assault','Criminal Damage','Robbery','Theft'])):
    ax = axes[idx//2, idx%2]
    monthly[crime].plot(kind='bar', ax=ax, color=colors[idx], edgecolor='black')
    ax.set_title(f'Average Daily {title} by Month', fontweight='bold')
plt.suptitle('Monthly Crime Seasonality in NYC', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{base_path}/outputs/monthly_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Day of week
merged['dow'] = pd.Categorical(merged['date'].dt.day_name(), categories=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'], ordered=True)
dow_avg = merged.groupby('dow', observed=False)[['assault','criminal_damage','robbery','theft']].mean()
fig, ax = plt.subplots(figsize=(12,6))
dow_avg.plot(kind='bar', ax=ax, color=colors, edgecolor='black')
ax.set_title('Average Daily Crime by Day of Week', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{base_path}/outputs/day_of_week.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top precincts
precinct_totals = merged.groupby('precinct')[['assault','criminal_damage','robbery','theft']].sum()
precinct_totals['total'] = precinct_totals.sum(axis=1)
top20 = precinct_totals.nlargest(20, 'total')
fig, ax = plt.subplots(figsize=(14,8))
top20[['assault','criminal_damage','robbery','theft']].plot(kind='barh', ax=ax, stacked=True, color=colors, edgecolor='black')
ax.set_title('Top 20 Precincts by Total Crime', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{base_path}/outputs/top_precincts.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
daily_agg = merged.groupby('date').agg({'assault':'sum','criminal_damage':'sum','robbery':'sum','theft':'sum','temperature':'first','humidity':'first','wind_speed':'first','precipitation':'first','discomfort_index':'first'}).reset_index()
corr = daily_agg[['assault','criminal_damage','robbery','theft','temperature','humidity','wind_speed','precipitation','discomfort_index']].corr()
fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, square=True)
ax.set_title('Correlation: Crime vs Weather', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{base_path}/outputs/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Adjacency graph
!pip install networkx -q
import networkx as nx
adj = np.load(f'{base_path}/processed_data/adjacency_matrix.npy')
pids = np.load(f'{base_path}/processed_data/precinct_ids.npy')
G = nx.relabel_nodes(nx.from_numpy_array(adj), {i: int(pids[i]) for i in range(len(pids))})
G.remove_edges_from(nx.selfloop_edges(G))
fig, ax = plt.subplots(figsize=(14,10))
nx.draw(G, nx.spring_layout(G, seed=42, k=2), ax=ax, with_labels=True, node_color='lightblue', node_size=500, font_size=8)
ax.set_title('NYC Precinct Adjacency Graph', fontsize=16, fontweight='bold')
plt.savefig(f'{base_path}/outputs/adjacency_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")